# Tumulus LiDAR Detector

Detects burial mounds (tumuli) in Romania's 0.5 m airborne LiDAR, in your browser. No install.

## To start: open the Runtime menu, click Run all, and wait a minute or two.

It sets up and shows a map; a default area is scanned automatically. Then just click anywhere in the blue on the map to scan that spot. The blue is where the model can scan (0.5 m LiDAR, ANCPI LAKI III; mostly Oltenia and SW Romania).

<img src="https://raw.githubusercontent.com/ObuObuHub/tumulus-lidar-detector/main/assets/coverage_preview.png" width="460">

Form is not proof: only fieldwork confirms a tumulus. Repo: https://github.com/ObuObuHub/tumulus-lidar-detector

In [ ]:
#@title Setup (run once) { display-mode: "form" }
import os
if not os.path.isdir('tumulus-lidar-detector') and os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    !git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
if os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    %cd tumulus-lidar-detector
!pip install -q pyproj
print("Ready. See the map below.")

In [ ]:
#@title Map - click anywhere in the blue to scan that spot { display-mode: "form" }
import os, sys, json, base64, subprocess
import IPython
from IPython.display import display, HTML
from google.colab import output as _colab_output

area_km = 1  #@param {type:"slider", min:1, max:5, step:1}

_cov = json.load(open('assets/laki3_coverage.geojson'))

def _inside(lon, lat):
    for f in _cov['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

def _scan(lat, lon):
    lat = float(lat); lon = float(lon)
    if not _inside(lon, lat):
        return IPython.display.JSON({'ok': False, 'msg': 'That point is outside the blue 0.5 m coverage. Click inside the blue area.'})
    for _f in ('/tmp/zone_dets.csv', 'review/zone_view.jpg', 'review/zone_board.jpg', 'detected_mounds.csv'):
        if os.path.exists(_f):
            os.remove(_f)
    subprocess.run([sys.executable, 'tools/scan_zone.py', str(lon), str(lat), str(area_km)], check=False)
    if not os.path.exists('review/zone_view.jpg'):
        return IPython.display.JSON({'ok': False, 'msg': 'No LiDAR tiles at this point.'})
    import pandas as pd
    n = 0; rows = []
    if os.path.exists('/tmp/zone_dets.csv'):
        det = pd.read_csv('/tmp/zone_dets.csv')
        kept = det[det['keep'] == 1].sort_values('score', ascending=False)
        n = len(kept)
        for i, (_, r) in enumerate(kept.iterrows(), 1):
            rows.append({'id': i, 'lon': round(float(r.lon), 6), 'lat': round(float(r.lat), 6),
                         'score': round(float(r.score), 3), 'coh': round(float(r.coh), 3),
                         'pgate': round(float(r.pgate), 3)})
    img = base64.b64encode(open('review/zone_view.jpg', 'rb').read()).decode()
    return IPython.display.JSON({'ok': True, 'n': n, 'img': img, 'rows': rows})

_colab_output.register_callback('notebook.scan', _scan)

_html = '''
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" integrity="sha384-sHL9NAb7lN7rfvG5lfHpm643Xkcjzp4jFvuavGOndn6pjVqS6ny56CAt3nsEVT4H" crossorigin="anonymous"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js" integrity="sha384-cxOPjt7s7Iz04uaHJceBmS+qpjv2JkIHNVcuOrM+YHwZOmJGBXI00mdUXEq65HTH" crossorigin="anonymous"></script>
<div id="tmap" style="height:460px;border:1px solid #ccc;border-radius:6px;"></div>
<div id="tinfo" style="font:14px/1.4 sans-serif;margin:10px 0;color:#111;"><b>Click anywhere in the blue area</b> to scan that spot (about a minute).</div>
<div id="tout"></div>
<script>
(function(){
  var cov = __COV__;
  function start(){
    if(typeof L === 'undefined'){ setTimeout(start, 200); return; }
    var map = L.map('tmap').setView([44.4, 23.6], 8);
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {attribution: 'OpenStreetMap'}).addTo(map);
    L.geoJSON(cov, {style: {color: '#1d4ed8', fillColor: '#2563eb', fillOpacity: 0.30, weight: 1}}).addTo(map);
    var marker = null, busy = false;
    async function scan(lat, lon){
      if(busy) return; busy = true;
      if(marker) map.removeLayer(marker);
      marker = L.marker([lat, lon]).addTo(map);
      document.getElementById('tinfo').innerHTML = 'Scanning ' + lat.toFixed(4) + ', ' + lon.toFixed(4) + ' ... about a minute.';
      document.getElementById('tout').innerHTML = '';
      try {
        var res = await google.colab.kernel.invokeFunction('notebook.scan', [lat, lon], {});
        var d = res.data['application/json'];
        if(!d.ok){ document.getElementById('tinfo').innerHTML = d.msg; busy = false; return; }
        document.getElementById('tinfo').innerHTML = d.n > 0 ? ('Found ' + d.n + ' probable mound(s).') : 'No mounds found here (normal for most points; the model is selective).';
        var html = '<img style="max-width:100%;border-radius:4px" src="data:image/jpeg;base64,' + d.img + '"/>';
        if(d.rows.length){
          html += '<table style="border-collapse:collapse;font:13px sans-serif;margin-top:10px"><tr style="background:#eee"><th style="border:1px solid #ccc;padding:4px 8px">id</th><th style="border:1px solid #ccc;padding:4px 8px">lon</th><th style="border:1px solid #ccc;padding:4px 8px">lat</th><th style="border:1px solid #ccc;padding:4px 8px">score</th><th style="border:1px solid #ccc;padding:4px 8px">map</th></tr>';
          d.rows.forEach(function(r){ html += '<tr><td style="border:1px solid #ccc;padding:4px 8px">' + r.id + '</td><td style="border:1px solid #ccc;padding:4px 8px">' + r.lon + '</td><td style="border:1px solid #ccc;padding:4px 8px">' + r.lat + '</td><td style="border:1px solid #ccc;padding:4px 8px">' + r.score + '</td><td style="border:1px solid #ccc;padding:4px 8px"><a target="_blank" href="https://www.google.com/maps?q=' + r.lat + ',' + r.lon + '">view</a></td></tr>'; });
          html += '</table>';
        }
        document.getElementById('tout').innerHTML = html;
      } catch(err){ document.getElementById('tinfo').innerHTML = 'Error: ' + err; }
      busy = false;
    }
    map.on('click', function(e){ scan(e.latlng.lat, e.latlng.lng); });
    scan(44.043, 23.522);   // default scan so something shows on Run all
  }
  start();
})();
</script>
'''
display(HTML(_html.replace('__COV__', json.dumps(_cov))))